# Qdrant × Future AGI: Fixing RAG Decay

This demo starts with a broken RAG app on a live Qdrant vector search engine. Each section shows one retrieval failure, applies one Qdrant fix, and checks that the metric moves.

- **App** (`app.py`): show the failure and the recovered answer
- **Notebook**: apply the fixes
- **Future AGI**: trace, judge, and compare runs

The notebook prints quick retrieval checks against labeled correct documents. Judged answer quality lives in Future AGI.

## 0 · Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

import agent
from helpers import config, embeddings
from helpers.embeddings import dense, sparse, colbert

load_dotenv()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = config.COLLECTION

# Start every run from the broken baseline: small dense model, no filter.
agent.set_retrieval(mode="minilm", current_only=False)
embeddings.warmup()   # load all models now, never during a live question
client.count(COLLECTION)

### Future AGI Tracing

`agent.py` registers tracing once:

```python
trace_provider = register(project_type=ProjectType.OBSERVE, project_name="pokedex-rag")
LangChainInstrumentor().instrument(tracer_provider=trace_provider)
```

Any process that imports the agent sends traces to `pokedex-rag`: this notebook, the Streamlit app, and rehearsal scripts. Qdrant calls appear as retriever spans.

## 1 · The Pain: Retrieval Decayed As The Corpus Grew

We score the same golden queries at three corpus sizes: Gen 1 only, Gen 1-4, and the full dex. `scaling_curve.py` generated the chart from the documents that existed at each stage.

Recall@5 falls 0.67 → 0.39 as lookalike species pile up. Duplicate rate is high from day one; growth makes the hidden flaws visible.

The curve uses all scored queries. Later sections use slices of the same golden set, so their baselines differ.

In [ ]:
from IPython.display import Image
Image("data/scaling_curve.png")   # generated offline by scaling_curve.py

## 2 · Fix #1: Dedup The Collection

**Dashboard read:** groundedness and context relevance can still look fine, but the retrieval panel shows the same `doc_id` filling the top-k. That points to duplication, not a model problem.

In [ ]:
# Duplicate rate in the top-k before the fix: a slot is wasted if it repeats a fragment
# with the same doc_id and text we already retrieved.
def dup_rate(query):
    chunks = agent.retrieve(query, mode="minilm")
    keys = [(c["doc_id"], c["text"]) for c in chunks]
    return keys, 1 - len(set(keys)) / len(keys)

keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")
for k in keys: print(" ", k[0])

In [ ]:
# THE FIX: delete duplicate points with the same doc_id + chunk_index, and keep one copy.
# The small viz collection gets the same dedup so the Web UI point cloud thins out too.
# (Reversible: `uv run python snapshot.py restore` brings the broken baseline back.)
from helpers.dedup import find_duplicate_ids

for col in (COLLECTION, config.VIZ_COLLECTION):
    before = client.count(col).count
    dup_ids = find_duplicate_ids(client, col)
    client.delete(collection_name=col, points_selector=dup_ids)
    print(f"{col}: {before} -> {client.count(col).count} points ({len(dup_ids)} duplicates deleted)")

In [ ]:
keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")   # now distinct documents fill the top-k
for k in keys: print(" ", k[0])

**Qdrant Web UI:** `pokemon_viz` in Visualize showed duplicates as tight clusters. Dedup cleaned that collection too, so refresh the view and the clusters thin out.

Dedup removes exact copies. Near-duplicate content can still survive, such as flavor text from several generations for the same species. That is a query-time diversity problem: use `group_by` or MMR when you need one result per document group.

## 3 · Fix #2: Upgrade The Embedding Model

**Dashboard read:** duplicates are gone, but queries that describe a Pokemon without naming it still miss. The small 384d model returns lookalikes: ask about Pikachu's cheek pouches and it retrieves Togedemaru.

The model did not suddenly get worse. The corpus grew until the old vector space stopped separating close neighbors.

Qdrant [named vectors](https://qdrant.tech/documentation/manage-data/vectors/#named-vectors) make the migration safe: add the candidate model to the live collection, A/B it on the golden set, then commit or roll back by flipping the retrieval mode.

The migration itself ran offline before the session (`prep.py`), because adding a vector name is instant but backfilling 23k points is slow:

```python
client.create_vector_name(COLLECTION, "dense_strong",
    vector_name_config=models.DenseVectorNameConfig(
        dense=models.DenseVectorConfig(size=1024, distance=models.Distance.COSINE)))
client.update_vectors(COLLECTION, points=[...])   # backfill; payloads untouched
```

In [ ]:
# Point the agent at the candidate model — one line, instantly reversible.
agent.set_retrieval(mode="bge")

# A/B both models on the golden queries this failure targets.
rows = [json.loads(line) for line in Path("data/golden_dataset.jsonl").read_text().splitlines() if line.strip()]
fix2 = [r for r in rows if r["exercises"] == "fix2_embedding"]

def recall_at_5(mode):
    """Share of queries with a correct document in the top 5 (unique docs, since
    fragments of one doc can fill several slots). Same scorer as verify_arc.py."""
    hits = 0
    for row in fix2:
        docs = []
        for chunk in agent.retrieve(row["query"], mode=mode, limit=30):
            if chunk["doc_id"] not in docs:
                docs.append(chunk["doc_id"])
        if set(docs[:5]) & set(row["gold_doc_ids"]):
            hits += 1
    return hits / len(fix2)

print(f"recall@5  small (MiniLM 384d)     = {recall_at_5('minilm'):.2f}")
print(f"recall@5  large (bge-large 1024d) = {recall_at_5('bge'):.2f}")

In [ ]:
# The eval confirms the win (0.64 -> 1.00 on these queries): keep bge.
# If the number had gone the other way, rollback is the same one-line flip.
print("retrieval mode:", agent.retrieval_state()["mode"])

The measurement is the point: the model that worked at 151 species stopped working at 1,025. Named vectors keep both models live, so the decision is reversible and based on this repo's golden set.

## 4 · Fix #3: Hybrid Retrieval + Reranking

**Dashboard read:** the strong dense model still misses the hardest paraphrases. Groundedness may stay green because the agent refuses correctly, so recall against labeled documents exposes the gap.

The fix is one [`query_points`](https://qdrant.tech/documentation/search/hybrid-queries/) call: fetch candidates with dense and sparse search, fuse the lists with RRF, then rerank the pool with ColBERT. It builds on the bge migration; it does not replace it.

In [ ]:
q = "A Pokemon that lives in dark caves and uses sound waves to navigate and hunt."

# Dense-only — even the strong model — misses it: nothing in the query names Zubat,
# and the cave/echolocation wording is spread across many bat- and cave-dweller entries.
dense_only = [c["doc_id"] for c in agent.retrieve(q, mode="bge")]
print("dense-only   :", dense_only)

# The hybrid + rerank call, spelled out. This is exactly what mode='hybrid' runs (agent.py).
dense_query = dense([q], config.MODEL_DENSE_STRONG, is_query=True)[0]
indices, values = sparse([q], is_query=True)[0]
colbert_query = colbert([q], is_query=True)[0]

candidates = models.Prefetch(
    prefetch=[
        models.Prefetch(query=dense_query, using=config.DENSE_STRONG,
                        limit=20, params=agent.EXACT),
        models.Prefetch(query=models.SparseVector(indices=indices, values=values),
                        using=config.SPARSE, limit=20),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),   # fuse dense + sparse lists
    limit=20,
)
hybrid = client.query_points(
    COLLECTION,
    prefetch=[candidates],
    query=colbert_query,          # rerank the 20 candidates with ColBERT
    using=config.COLBERT,
    limit=5,
    with_payload=["doc_id"],
).points
print("hybrid+rerank:", [h.payload["doc_id"] for h in hybrid])

In [ ]:
# Flip the agent to hybrid, then score the queries dense retrieval was missing.
agent.set_retrieval(mode="hybrid")

import math
fix3 = [r for r in rows if r["exercises"] == "fix3_hybrid"]

def score(mode):
    """recall@5, NDCG@5, MRR against the labeled documents (each query has one gold doc,
    so the ideal DCG is 1). Same scorer as verify_arc.py."""
    recall, ndcg, mrr = 0, 0, 0
    for row in fix3:
        docs = []
        for chunk in agent.retrieve(row["query"], mode=mode, limit=30):
            if chunk["doc_id"] not in docs:
                docs.append(chunk["doc_id"])
        top5, gold = docs[:5], set(row["gold_doc_ids"])
        recall += any(d in gold for d in top5)
        ndcg += sum(1 / math.log2(i + 2) for i, d in enumerate(top5) if d in gold)
        mrr += next((1 / (i + 1) for i, d in enumerate(top5) if d in gold), 0)
    n = len(fix3)
    return recall / n, ndcg / n, mrr / n

print("dense (bge)   recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("bge"))
print("hybrid+rerank recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("hybrid"))

Why both halves matter, measured stage by stage: prefetch widens the candidate pool, so at rank 20 it holds the right document for 94% of queries. But fusion alone scores worse at top-5 (0.72) than pure dense (0.78), because a wider pool is noisier. The ColBERT rerank turns that pool into wins: recall@5 0.78 → 0.89, NDCG@5 0.61 → 0.77. Prefetch builds the pool; the reranker orders it.

One caveat: this scored set is deliberately all paraphrases, the queries dense search struggles with. Sparse search earns its place on everyday entity and keyword queries that are not scored here.

## 5 · Close The Cold Open With Metadata

Dedup, bge, and hybrid still do not fix the opening failure. The stale pre-Gen-6 chart matches the question too well, so it keeps winning retrieval.

Better ranking cannot fix wrong-but-relevant data. The points already carry `is_current`; index it and filter on it.

In [ ]:
q = "Does the Steel type resist Ghost and Dark attacks?"
before = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("no filter :", before)

# THE FIX: index the is_current payload field, then have the agent filter on it.
client.create_payload_index(COLLECTION, "is_current", models.PayloadSchemaType.BOOL)
agent.set_retrieval(current_only=True)

after = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("is_current:", after)

In [ ]:
# Ask the agent the cold-open question again. Now it is grounded in the current document.
answer, _ = agent.ask("Does the Steel type resist Ghost and Dark attacks?")
print(answer)

## 6 · Bonus: Multi-Hop In The Trace View

A compound question makes the agent retrieve twice. In Future AGI, it should show up as two retriever spans from one user request.

In [ ]:
answer, chunks = agent.ask("What does Drowzee eat, and is that Pokemon weak to Bug-type attacks?")
print(answer)